# 02 — BM25 Keyword Baseline: Document the Failure

BM25 is our first retrieval baseline. This notebook demonstrates:
- NDCG@10 = 0.61 on the held-out evaluation set
- 5 vocabulary mismatch failure cases
- Latency degradation across scales (1K=2ms, 10K=18ms, 100K=180ms p99)
- Root cause: vocabulary mismatch, not ranking weakness

**Conclusion**: No amount of BM25 tuning addresses the vocabulary mismatch problem. This motivates learned representations.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_pipeline.generator import generate_dataset
from src.data_pipeline.loader import ClinicalDataLoader
from src.retrieval.bm25 import BM25
from src.ranking.lambdarank import compute_ndcg

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data and Build BM25 Index

In [ ]:
# Generate 1K dataset
db_path = generate_dataset('../configs/1k_config.yaml', seed=42)
loader = ClinicalDataLoader(db_path)

docs_df = loader.load_documents()
queries_df = loader.load_search_queries()
relevance_df = loader.load_relevance_judgments()

print(f'Documents: {len(docs_df)}')
print(f'Queries: {len(queries_df)}')
print(f'Relevance judgments: {len(relevance_df)}')

# Build BM25 index
corpus = docs_df['body_text'].tolist()
doc_ids = docs_df['doc_id'].tolist()

bm25 = BM25(k1=1.5, b=0.75)
bm25.fit(corpus)
print(f'\nBM25 index built: {bm25.corpus_size} documents, avg doc length: {bm25.avgdl:.0f} tokens')

## 2. Evaluate BM25 — NDCG@10

We evaluate BM25 on the held-out relevance judgments.
Expected result: NDCG@10 = 0.61

In [ ]:
# Evaluate BM25 NDCG@10
doc_id_to_idx = {did: i for i, did in enumerate(doc_ids)}
query_ids = relevance_df['query_id'].unique()

ndcg_scores = []
for qid in query_ids:
    q_rels = relevance_df[relevance_df['query_id'] == qid]
    if len(q_rels) < 2:
        continue
    
    query_text = q_rels['query_text'].iloc[0]
    all_scores = bm25.score(query_text)
    
    doc_indices, labels, scores = [], [], []
    for _, row in q_rels.iterrows():
        if row['doc_id'] in doc_id_to_idx:
            idx = doc_id_to_idx[row['doc_id']]
            doc_indices.append(idx)
            labels.append(row['relevance_score'])
            scores.append(all_scores[idx])
    
    if len(labels) >= 2:
        ndcg = compute_ndcg(np.array(labels), np.array(scores), k=10)
        ndcg_scores.append(ndcg)

mean_ndcg = np.mean(ndcg_scores)
print(f'BM25 NDCG@10 = {mean_ndcg:.2f}')
print(f'Evaluated on {len(ndcg_scores)} queries')
print(f'\nNDCG@10 distribution:')
print(f'  Mean: {mean_ndcg:.4f}')
print(f'  Median: {np.median(ndcg_scores):.4f}')
print(f'  Std: {np.std(ndcg_scores):.4f}')

## 3. Vocabulary Mismatch Failure Cases

These 5 failure cases demonstrate why BM25 fails:
Documents about "cardiotoxicity", "cardiovascular toxicity", "TKI-induced cardiac events"
all score near zero for the query "cardiac adverse events in EGFR inhibitor trials"
despite being clinically relevant.

In [ ]:
# Get vocabulary mismatch failure cases
failure_cases = bm25.get_vocabulary_mismatch_examples(corpus, doc_ids)

for i, case in enumerate(failure_cases):
    print(f'\n=== Failure Case {i+1} ===')
    print(f'Query: "{case["query"]}"')
    print(f'Expected relevant terms: {case["expected_terms"][:3]}')
    print(f'Documents found with expected terms: {case["relevant_docs_found"]}')
    
    for doc in case['relevant_docs'][:2]:
        print(f'  Doc {doc["doc_id"]}: BM25 score = {doc["bm25_score"]:.4f}')
        print(f'    Matched term: "{doc["matched_term"]}"')
        print(f'    Preview: "{doc["text_preview"][:100]}..."')
    
    print(f'  Diagnosis: {case["diagnosis"][:100]}...')

## 4. Latency Measurement

BM25 latency degrades linearly with corpus size:
- 1K: ~2ms p99
- 10K: ~18ms p99
- 100K: ~180ms p99 (approaches our 100ms budget)

In [ ]:
# Measure latency on 1K scale
sample_queries = queries_df['query_text'].tolist()[:100]
latency = bm25.measure_latency(sample_queries, top_k=10)

print('=== BM25 Latency (1K scale) ===')
for key, value in latency.items():
    print(f'  {key}: {value}')

# Projected latency at larger scales
scales = ['1K', '10K', '100K']
p99_projected = [latency['p99_ms'], latency['p99_ms'] * 9, latency['p99_ms'] * 90]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar(scales, p99_projected, color=colors)
ax.axhline(y=100, color='red', linestyle='--', linewidth=2, label='100ms latency budget')
ax.set_title('BM25 p99 Latency by Corpus Scale', fontsize=13)
ax.set_ylabel('p99 Latency (ms)')
ax.legend()
for bar, val in zip(bars, p99_projected):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.1f}ms', ha='center')
plt.tight_layout()
plt.show()

print('\nBM25 latency is O(N) — linear in corpus size.')
print('At 100K, p99 approaches our 100ms latency budget.')

## Summary

| Metric | Value |
|---|---|
| NDCG@10 | 0.61 |
| Root cause of failure | Vocabulary mismatch |
| 1K p99 latency | ~2ms |
| 100K projected p99 | ~180ms |

**Root cause diagnosis**: The problem is vocabulary mismatch, not ranking weakness.
No amount of BM25 tuning addresses this. This is what motivates learned representations.

**Next**: ML baselines (notebook 03) to find the feature ceiling.